In [1]:
import scipy
scipy.stats.binom.cdf(100, 1000, 0.3)

np.float64(1.7183404003299047e-52)

In [2]:
from scipy.stats import norm

probability = norm.cdf(3) - norm.cdf(-3)

print("Доверительный интервал:", probability) 
# Выведет: 0.9973002039367398 (то есть 99.73%)

print("Уровень значимости (1 - p):", 1 - probability)

Доверительный интервал: 0.9973002039367398
Уровень значимости (1 - p): 0.002699796063260207


In [5]:
from scipy.stats import t

t.interval(0.95, df=24, loc=170, scale=12 / (25**0.5))

(np.float64(165.04664345209275), np.float64(174.95335654790725))

In [3]:
scipy.stats.t.ppf(1-0.05/2, 24)

np.float64(2.0638985616280205)

In [6]:
import numpy as np
data = np.random.poisson(lam=50, size=1000)
# Считаем параметры выборки
mean = np.mean(data)
se = np.std(data, ddof=1) / np.sqrt(1000)

# Вычисляем границы и ширину
low, high = norm.interval(0.95, loc=mean, scale=se)
width = high - low
print("Ширина интервала:", width)

Ширина интервала: 0.8994916319279014


In [7]:
import numpy as np
from scipy.stats import bootstrap

# Генерируем ту же самую выборку из Пуассона
np.random.seed(42) # зафиксируем для примера
data = np.random.poisson(lam=50, size=1000)

# Бутстрэп для стандартного отклонения (std)
res = bootstrap((data,), np.std, confidence_level=0.95, method='percentile', n_resamples=2000)
print(f"Интервал: [{res.confidence_interval.low:.2f}, {res.confidence_interval.high:.2f}]")
print(f"Ширина: {res.confidence_interval.high - res.confidence_interval.low:.2f}")

Интервал: [6.97, 7.57]
Ширина: 0.60


In [8]:
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True, as_frame=True)

In [9]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error


# Зафиксируем сид для воспроизводимости (попробуйте убрать или изменить, если числа не сойдутся)
np.random.seed(42)

n_samples = len(X)
mse_scores = []
n_iterations = 1000
all_indices = np.arange(n_samples)

# 2. Проводим бутстрэп-валидацию
for _ in range(n_iterations):
    # Генерация индексов с возвращением
    train_indices = np.random.choice(all_indices, size=n_samples, replace=True)
    # Выделение оставшихся объектов (Out-of-Bag)
    test_indices = np.setdiff1d(all_indices, train_indices)
    
    # Разделение на тренировочную и тестовую выборки
    X_train, y_train = X.iloc[train_indices], y.iloc[train_indices]
    X_test, y_test = X.iloc[test_indices], y.iloc[test_indices]
    
    # Обучение базовой Ridge без предобработки данных
    model = Ridge()
    model.fit(X_train, y_train)
    
    # Оценка качества на отложенных объектах
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    mse_scores.append(mse)

# 3. Вычисление перцентилей для 95% доверительного интервала
lower_bound = np.percentile(mse_scores, 2.5)
upper_bound = np.percentile(mse_scores, 97.5)

print(f"Результат: [{lower_bound:.3f}, {upper_bound:.3f}]")

Результат: [0.505, 1.156]


In [10]:
import numpy as np
from scipy.stats import norm

# Параметры по условию
true_mean = 3
true_std = 4
n_samples = 1000
n_experiments = 1000

covered_count = 0
z_crit = norm.ppf(0.975) # Квантиль для 95% ДИ (~1.96)

for _ in range(n_experiments):
    # Генерация случайной выборки
    sample = np.random.normal(loc=true_mean, scale=true_std, size=n_samples)
    
    # Оценка параметров по выборки для ЦПТ
    sample_mean = np.mean(sample)
    se = np.std(sample, ddof=1) / np.sqrt(n_samples)
    
    # Строим 95% доверительный интервал
    lower_bound = sample_mean - z_crit * se
    upper_bound = sample_mean + z_crit * se
    
    # Проверяем, попало ли истинное среднее (3) в интервал
    if lower_bound <= true_mean <= upper_bound:
        covered_count += 1

share_of_coverage = covered_count / n_experiments
print(f"Доля случаев покрытия: {share_of_coverage}") # Вы

Доля случаев покрытия: 0.949
